# 01 — Decision Tree
Notebook thực nghiệm độc lập cho SecuraAI / Windows-APT 2025. Mọi lựa chọn được khóa bằng Train/Validation trước khi đánh giá Test.

## 0. Chuẩn bị môi trường
Chạy từ thư mục project hoặc thư mục `notebooks`. Không cần GPU.

In [ ]:
from pathlib import Path
import json, hashlib
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import platform, sklearn
import importlib
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    accuracy_score, precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
    average_precision_score, confusion_matrix, make_scorer)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.dummy import DummyClassifier
from securaai_ml.data import load_training_data
import securaai_ml.modeling as modeling_module
importlib.reload(modeling_module)  # tránh module cache cũ khi source vừa được cập nhật
from securaai_ml.modeling import (candidate_factories, decision_scores, parameter_grids,
    teaching_candidate_factories, teaching_parameter_grids)
from securaai_ml.schema import MODEL_FEATURES, TARGET_COLUMN, GROUP_COLUMN
from securaai_ml.train import temporal_train_validation_test

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
data, audit = load_training_data(ROOT / 'data' / 'raw')
train, validation, test = temporal_train_validation_test(data)
X_train, y_train = train.loc[:, MODEL_FEATURES], train[TARGET_COLUMN]
X_validation, y_validation = validation.loc[:, MODEL_FEATURES], validation[TARGET_COLUMN]
groups = train[GROUP_COLUMN]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
audit

### Tính tái lập
Cố định seed và in phiên bản. Đổi seed là một thí nghiệm mới, không chọn seed bằng kết quả Test.

In [ ]:
import platform
import numpy as np
import matplotlib.pyplot as plt
import sklearn

SEED=42
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi':110,'font.family':'DejaVu Sans','font.size':10})
print('Python:',platform.python_version(),'| sklearn:',sklearn.__version__,'| seed:',SEED)

## 1. Xác thực dữ liệu và ý nghĩa nhãn
Nhãn MALICIOUS_APT được suy ra từ MITRE/Wazuh tag, không phải ground truth do chuyên gia xét duyệt. Các cột tạo nhãn bị loại khỏi đầu vào.

In [ ]:
display(pd.Series(audit.__dict__,name='value').to_frame())
assert audit.rows==len(data)
assert data[TARGET_COLUMN].nunique()==2
assert data[GROUP_COLUMN].ne('unknown-date').all()
print('Dataset checks passed')

## 2. Chia Train/Validation/Test và dấu vân tay
Chia xấp xỉ 60/20/20 theo ngày; không có ngày nào xuất hiện ở hai tập.

In [ ]:
split_table=pd.DataFrame([{'set':name,'rows':len(part),'benign':int((part[TARGET_COLUMN]==0).sum()),'malicious':int((part[TARGET_COLUMN]==1).sum()),'attack_rate':float(part[TARGET_COLUMN].mean()),'dates':part[GROUP_COLUMN].nunique()} for name,part in [('Train',train),('Validation',validation),('Test',test)]])
display(split_table)
assert set(train[GROUP_COLUMN]).isdisjoint(validation[GROUP_COLUMN])
assert set(train[GROUP_COLUMN]).isdisjoint(test[GROUP_COLUMN])
assert set(validation[GROUP_COLUMN]).isdisjoint(test[GROUP_COLUMN])
split_hash=hashlib.sha256(json.dumps({'train':sorted(train[GROUP_COLUMN].unique()),'validation':sorted(validation[GROUP_COLUMN].unique()),'test':sorted(test[GROUP_COLUMN].unique())},sort_keys=True).encode()).hexdigest()
print('Split hash:',split_hash[:16])

## 3. EDA chỉ trên Train
Quan sát mất cân bằng và tỷ lệ tấn công theo nhóm sự kiện mà không dùng Test để định hướng mô hình.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(13,4))
train[TARGET_COLUMN].map({0:'BENIGN',1:'MALICIOUS_APT'}).value_counts().plot.bar(ax=ax[0],rot=0,color=['#9bb5c5','#087f8c']); ax[0].set_title('Phân bố nhãn — TRAIN'); ax[0].set_ylabel('Số log')
provider_rate=train.groupby('provider')[TARGET_COLUMN].agg(['count','mean']).query('count >= 100').sort_values('count',ascending=False).head(12)
provider_rate['mean'].plot.bar(ax=ax[1],color='#087f8c'); ax[1].set_title('Tỷ lệ tấn công theo provider — TRAIN'); ax[1].set_ylabel('Tỷ lệ MALICIOUS_APT')
plt.tight_layout(); plt.show(); display(provider_rate)

## 4. Cơ chế thuật toán và phạm vi triển khai
Cây chia dữ liệu theo điều kiện feature. Dễ giải thích nhưng dễ overfit; tuning `max_depth` và `min_samples_leaf`.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
# Sơ đồ minh họa cơ chế chia nhánh; chưa phải cây đã học từ dataset.
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')
nodes = {
    'root': (0.50, 0.88, 'process_name = cmd.exe?'),
    'left': (0.25, 0.58, 'has_command_line = 1?'),
    'right': (0.75, 0.58, 'event_id đáng chú ý?'),
    'll': (0.12, 0.23, 'MALICIOUS_APT'),
    'lr': (0.38, 0.23, 'BENIGN'),
    'rl': (0.62, 0.23, 'MALICIOUS_APT'),
    'rr': (0.88, 0.23, 'BENIGN'),
}
for key_name, (x, y, label) in nodes.items():
    leaf = key_name in {'ll','lr','rl','rr'}
    color = '#e76f51' if label == 'MALICIOUS_APT' else ('#2a9d8f' if label == 'BENIGN' else '#d9eaf0')
    ax.text(x, y, label, ha='center', va='center', fontsize=11,
            bbox=dict(boxstyle='round,pad=0.55', facecolor=color, edgecolor='#264653'))
for parent, child, branch in [('root','left','Có'),('root','right','Không'),('left','ll','Có'),('left','lr','Không'),('right','rl','Có'),('right','rr','Không')]:
    x1,y1,_=nodes[parent]; x2,y2,_=nodes[child]
    ax.annotate('', xy=(x2,y2+0.055), xytext=(x1,y1-0.055), arrowprops=dict(arrowstyle='->',lw=1.5,color='#555'))
    ax.text((x1+x2)/2,(y1+y2)/2,branch,fontsize=9,ha='center',backgroundcolor='white')
ax.set_title('Minh họa cơ chế Decision Tree (không phải luật thật của model)', fontsize=14)
plt.show()

**Cách đọc:** bắt đầu từ nút trên cùng, kiểm tra điều kiện rồi đi theo nhánh Có/Không cho đến nút lá. Sau khi mô hình được huấn luyện, mục 9 sẽ hiển thị cây thật và các feature mà model thực sự sử dụng.

## 5. Tiền xử lý an toàn trong Pipeline
One-Hot Encoder được fit bên trong từng fold. Không fit preprocessing trên toàn bộ dữ liệu trước CV.

In [ ]:
from sklearn import set_config
from IPython.display import display

set_config(display='diagram')
candidate=teaching_candidate_factories(SEED)['decision_tree']()
grid=teaching_parameter_grids()['decision_tree']

# Trong Jupyter, object Pipeline cuối cell được render thành sơ đồ HTML có thể mở rộng.
display(candidate)

grid_table=pd.DataFrame([
    {'hyperparameter':name, 'candidate_values':str(values), 'purpose':(
        'Giới hạn độ sâu, kiểm soát độ phức tạp' if 'max_depth' in name else
        'Số mẫu tối thiểu ở mỗi lá, hạn chế overfit' if 'min_samples_leaf' in name else
        'Giá trị được Grid Search thử nghiệm')}
    for name,values in grid.items()
])
display(grid_table)

## 6. Độ đo và Dummy baseline
Accuracy có thể gây hiểu nhầm. F2 nhấn mạnh Recall; ROC-AUC đo xếp hạng; AP hữu ích khi mất cân bằng.

In [ ]:
dummy=DummyClassifier(strategy='prior').fit(X_train,y_train)
dummy_pred=dummy.predict(X_validation); dummy_score=dummy.predict_proba(X_validation)[:,1]
display(pd.Series({'accuracy':accuracy_score(y_validation,dummy_pred),'precision':precision_score(y_validation,dummy_pred,zero_division=0),'recall':recall_score(y_validation,dummy_pred,zero_division=0),'f1':f1_score(y_validation,dummy_pred,zero_division=0),'f2':fbeta_score(y_validation,dummy_pred,beta=2,zero_division=0),'roc_auc':roc_auc_score(y_validation,dummy_score),'average_precision':average_precision_score(y_validation,dummy_score)},name='Dummy — Validation').to_frame())

## 7. Chọn siêu tham số bằng 5-Fold chỉ trên Train
Grid Search chấm F2, F1, ROC-AUC và AP; `refit='f2'` khóa tiêu chí lựa chọn trước khi xem Test.

In [ ]:
model_name = 'decision_tree'
search = GridSearchCV(
    teaching_candidate_factories(42)[model_name](),
    teaching_parameter_grids()[model_name],
    scoring={'f2': make_scorer(fbeta_score, beta=2, zero_division=0),
             'f1': 'f1', 'roc_auc': 'roc_auc', 'average_precision': 'average_precision'},
    refit='f2',
    cv=cv, n_jobs=1, return_train_score=True,
)
search.fit(X_train, y_train, groups=groups)
print('Best parameters:', search.best_params_)
print('Mean CV F2:', search.best_score_)
model = search.best_estimator_

In [ ]:
cv_table=pd.DataFrame(search.cv_results_).sort_values('rank_test_f2')
display(cv_table[['params','mean_train_f2','mean_test_f2','std_test_f2','mean_test_roc_auc','mean_test_average_precision']].head(12))

## 8. Đánh giá trên Validation
Confusion Matrix cho biết TN/FP/FN/TP. ROC và PR phải dùng score liên tục, không dùng nhãn 0/1 đã cắt.

In [ ]:
pred = model.predict(X_validation)
score = decision_scores(model, X_validation)
result = pd.Series({
    'accuracy': accuracy_score(y_validation, pred),
    'precision': precision_score(y_validation, pred, zero_division=0),
    'recall': recall_score(y_validation, pred, zero_division=0),
    'f1': f1_score(y_validation, pred, zero_division=0),
    'f2': fbeta_score(y_validation, pred, beta=2, zero_division=0),
    'roc_auc': roc_auc_score(y_validation, score),
    'average_precision': average_precision_score(y_validation, score),
})
display(result.to_frame('Validation'))
tn, fp, fn, tp = confusion_matrix(y_validation, pred, labels=[0, 1]).ravel()
fpr_at_threshold = fp / (fp + tn)
tpr_at_threshold = tp / (tp + fn)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
ConfusionMatrixDisplay.from_predictions(y_validation, pred, display_labels=['BENIGN','MALICIOUS_APT'], ax=axes[0])
axes[0].set_title('Validation — Confusion Matrix')
RocCurveDisplay.from_predictions(y_validation, score, name=model_name.replace('_', ' ').title(), ax=axes[1])
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Random')
axes[1].scatter([fpr_at_threshold], [tpr_at_threshold], color='darkorange', zorder=5, label='Decision threshold')
axes[1].set_title('ROC — VALIDATION'); axes[1].legend()
PrecisionRecallDisplay.from_predictions(y_validation, score, name=model_name.replace('_', ' ').title(), ax=axes[2])
base_rate = float(y_validation.mean())
axes[2].axhline(base_rate, ls='--', color='gray', label=f'Positive rate={base_rate:.3f}')
axes[2].set_title('Precision–Recall — VALIDATION'); axes[2].legend()
plt.tight_layout(); plt.show()

## 9. Trực quan cây và diễn giải
Chỉ vẽ ba tầng đầu để đọc được; cây đầy đủ vẫn được giữ trong model.

In [ ]:
from sklearn.tree import plot_tree, export_text
pre=model.named_steps['preprocess']
tree=model.named_steps['model']
feature_names=pre.get_feature_names_out()
plt.figure(figsize=(22,9))
plot_tree(tree,max_depth=3,feature_names=feature_names,class_names=['BENIGN','MALICIOUS_APT'],filled=True,rounded=True,fontsize=7)
plt.title('Decision Tree — ba tầng đầu'); plt.show()
print(export_text(tree,feature_names=list(feature_names),max_depth=3))

## 10. Chọn và khóa ngưỡng trên Validation
Với score rời rạc, thử toàn bộ giá trị duy nhất; với score liên tục, thử các phân vị. Luôn bổ sung ngưỡng mặc định rồi chọn F2 cao nhất. Không điều chỉnh lại sau khi xem Test.

In [ ]:
validation_score=decision_scores(model,X_validation)
unique_scores=np.unique(validation_score)
default_threshold=0.5 if hasattr(model,'predict_proba') else 0.0
if len(unique_scores)<=1000:
    threshold_grid=unique_scores
    grid_strategy='all unique scores'
else:
    threshold_grid=np.unique(np.quantile(validation_score,np.linspace(0.001,0.999,999)))
    grid_strategy='score quantiles'
threshold_grid=np.unique(np.append(threshold_grid,default_threshold))
print(f'Score levels: {len(unique_scores):,} | strategy: {grid_strategy} | default: {default_threshold}')
threshold_rows=[]
for threshold in threshold_grid:
    p=(validation_score>=threshold).astype(int)
    threshold_rows.append({'threshold':float(threshold),'precision':precision_score(y_validation,p,zero_division=0),'recall':recall_score(y_validation,p,zero_division=0),'f1':f1_score(y_validation,p,zero_division=0),'f2':fbeta_score(y_validation,p,beta=2,zero_division=0)})
threshold_table=pd.DataFrame(threshold_rows).sort_values('threshold')
chosen=threshold_table.sort_values(['f2','recall','precision'],ascending=False).iloc[0]
locked_threshold=float(chosen.threshold)
display(threshold_table.sort_values('f2',ascending=False).head(10))
threshold_table.plot(x='threshold',y=['precision','recall','f1','f2'],figsize=(11,5)); plt.axvline(locked_threshold,color='black',ls='--',label=f'locked={locked_threshold:.4g}'); plt.legend(); plt.show()

## 11. Đánh giá Test đúng một lần
Siêu tham số và ngưỡng đã khóa. Kết quả Test chỉ dùng báo cáo cuối, không quay lại sửa thiết kế.

In [ ]:
X_test,y_test=test.loc[:,MODEL_FEATURES],test[TARGET_COLUMN]
test_score=decision_scores(model,X_test)
test_pred=(test_score>=locked_threshold).astype(int)
test_result=pd.Series({'accuracy':accuracy_score(y_test,test_pred),'precision':precision_score(y_test,test_pred,zero_division=0),'recall':recall_score(y_test,test_pred,zero_division=0),'f1':f1_score(y_test,test_pred,zero_division=0),'f2':fbeta_score(y_test,test_pred,beta=2,zero_division=0),'roc_auc':roc_auc_score(y_test,test_score),'average_precision':average_precision_score(y_test,test_score)})
display(test_result.to_frame('TEST'))
tn,fp,fn,tp=confusion_matrix(y_test,test_pred,labels=[0,1]).ravel()
fig,axes=plt.subplots(1,3,figsize=(17,4.8))
ConfusionMatrixDisplay.from_predictions(y_test,test_pred,display_labels=['BENIGN','MALICIOUS_APT'],ax=axes[0]); axes[0].set_title(f'TEST — locked threshold {locked_threshold:.4g}')
RocCurveDisplay.from_predictions(y_test,test_score,name=f"AUC={test_result['roc_auc']:.3f}",ax=axes[1]); axes[1].plot([0,1],[0,1],'--',color='gray'); axes[1].scatter([fp/(fp+tn)],[tp/(tp+fn)],color='darkorange',zorder=5,label='Locked threshold'); axes[1].set_title('ROC — TEST'); axes[1].legend()
PrecisionRecallDisplay.from_predictions(y_test,test_score,name=f"AP={test_result['average_precision']:.3f}",ax=axes[2]); axes[2].axhline(float(y_test.mean()),ls='--',color='gray',label=f'Attack rate={y_test.mean():.3f}'); axes[2].set_title('Precision–Recall — TEST'); axes[2].legend(); plt.tight_layout(); plt.show()

## 12. Khoảng bất định bằng bootstrap
Bootstrap trên Test chỉ để mô tả độ bất định, không huấn luyện hoặc chọn ngưỡng lại.

In [ ]:
rng=np.random.default_rng(SEED); boot=[]
y_array=y_test.to_numpy()
for _ in range(300):
    idx=rng.integers(0,len(y_array),len(y_array))
    if np.unique(y_array[idx]).size<2: continue
    boot.append({'f2':fbeta_score(y_array[idx],test_pred[idx],beta=2,zero_division=0),'roc_auc':roc_auc_score(y_array[idx],test_score[idx]),'average_precision':average_precision_score(y_array[idx],test_score[idx])})
boot=pd.DataFrame(boot)
display(pd.DataFrame({'estimate':test_result[['f2','roc_auc','average_precision']], 'ci_2.5%':boot.quantile(.025), 'ci_97.5%':boot.quantile(.975)}))

## 13. Lưu kết quả thực nghiệm
Lưu model, ngưỡng, split hash và metric để notebook so sánh tổng hợp có thể kiểm tra nguồn gốc.

In [ ]:
experiment_dir=ROOT/'reports'/'notebook_experiments'; experiment_dir.mkdir(parents=True,exist_ok=True)
slug='01_decision_tree'
joblib.dump(model,experiment_dir/f'{slug}.joblib')
record={'model':model_name,'best_params':search.best_params_,'cv_best_f2':float(search.best_score_),'locked_threshold':locked_threshold,'split_hash':split_hash,'validation':{k:float(v) for k,v in result.items()},'test':{k:float(v) for k,v in test_result.items()}}
(experiment_dir/f'{slug}.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
record

## 14. Kết luận
So sánh chính thức ở `08_Lua_Chon_Mo_Hinh_Va_Bao_Cao.ipynb`. Giới hạn: nhãn rule-derived, dữ liệu mô phỏng Windows APT và chưa phải bằng chứng hiệu năng production.